# 市场状态系统雏形：状态 + 资金

**发射分布按维度选**：价格维度 Student-t，波动维度 Gaussian。
**状态数**：短期 3×3（九宫格），长期 2×2（四宫格）——60–120 窗口下长期第三态几乎不存在。

后面几节是资金变量（期货基差/持仓、ETF 份额、两融、成交额），全部先做滚动 z 归一化。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "state_layer").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from experiment.hmm_state.matrix import _index_panel
from experiment.hmm_state.features import LONG_DEFAULT, build_groups, order_col_for, agreement
from experiment.hmm_state.final import K_OF, MODEL, fit_dim, grid
from experiment.hmm_state.money import load_signals, buckets, coverage, SIGNALS

pd.set_option("display.width", 160)
px.defaults.template = "plotly_white"

px_panel = _index_panel()
print("长期窗口:", LONG_DEFAULT, "| 状态数:", K_OF)
groups = build_groups(px_panel, LONG_DEFAULT)
groups = {n: f for n, f in groups.items() if n in MODEL}   # 中长期不建波动模型

fitted = {}
for name, feats in groups.items():
    fitted[name] = fit_dim(name, feats, order_col_for(name, LONG_DEFAULT), k=K_OF[name])
    print(f"{name:6s} K={K_OF[name]} {fitted[name]['model']:14s} NLL={fitted[name]['nll']:.4f} "
          f"占比={fitted[name]['share']}")

长期窗口: 60-120 | 状态数: {'短价格': 3, '短波动': 3, '长价格': 2}


短价格    K=3 Student-t(ν=5) NLL=1.2619 占比={0: 0.507, 1: 0.258, 2: 0.235}


短波动    K=3 Gaussian       NLL=0.2903 占比={0: 0.591, 1: 0.345, 2: 0.064}


长价格    K=2 Student-t(ν=5) NLL=1.7567 占比={0: 0.553, 1: 0.447}


In [2]:
PALETTE = ["#2E86C1", "#27AE60", "#C0392B", "#8E44AD", "#D68910", "#17A589"]
BASE_COLORS = ["#5499C7", "#E67E22", "#CB4335", "#7D3C98"]


def cell_colors(na, nb):
    colors, order = [], []
    for v in range(nb):
        base = BASE_COLORS[v % len(BASE_COLORS)]
        for p in range(na):
            k = 0.45 + 0.55 * (p / max(na - 1, 1))
            rgb = [int(int(base[i:i + 2], 16) * k) for i in (1, 3, 5)]
            colors.append("rgb(%d,%d,%d)" % tuple(rgb))
            order.append(f"{p}-{v}")
    return colors, order


def sizes(pk, vk):
    na = int(fitted[pk]["states"].max()) + 1
    nb = int(fitted[vk]["states"].max()) + 1
    return na, nb


def cells(pk, vk):
    return grid(fitted[pk]["states"], fitted[vk]["states"])

## 工具函数（先跑）

In [3]:
def plot_states(name):
    r = fitted[name]
    states = r["states"]
    close = px_panel["close"].reindex(states.index)
    prof = r["profile"]

    fig = make_subplots(rows=2, cols=1, shared_xaxes=False, row_heights=[0.68, 0.32],
                        vertical_spacing=0.12,
                        subplot_titles=(f"{name}（{r['model']}, K={len(prof)}）：价格按过滤状态着色",
                                        "状态画像（各特征均值）"))
    fig.add_trace(go.Scatter(x=close.index, y=close.values, mode="lines",
                             line=dict(color="#BDC3C7", width=1), name="价格",
                             showlegend=False), row=1, col=1)
    for i, s in enumerate(sorted(states.unique())):
        m = states == s
        seg = close.where(m)
        c = PALETTE[i % len(PALETTE)]
        fig.add_trace(go.Scatter(x=seg.index, y=seg.values, mode="lines+markers",
                                 line=dict(color=c, width=2.2),
                                 marker=dict(size=4, color=c),
                                 name=f"状态 {s}（{int(m.sum())} 天）"), row=1, col=1)
    for i, s in enumerate(sorted(prof.index)):
        fig.add_trace(go.Bar(x=prof.columns, y=prof.loc[s].values, name=f"状态 {s}",
                             marker_color=PALETTE[i % len(PALETTE)],
                             showlegend=False), row=2, col=1)
    fig.update_layout(height=620, margin=dict(l=40, r=20, t=60, b=40),
                      hovermode="x unified",
                      legend=dict(orientation="h", yanchor="bottom", y=1.02))
    fig.update_yaxes(title_text="价格", row=1, col=1)
    fig.update_yaxes(title_text="特征均值", row=2, col=1)
    return fig


def plot_grid(pk, vk, title):
    na, nb = sizes(pk, vk)
    colors, order = cell_colors(na, nb)
    cell = cells(pk, vk)
    close = px_panel["close"].reindex(cell.index)
    share = cell.value_counts(normalize=True).reindex(order).fillna(0) * 100

    heat = np.zeros((nb, na))
    for c, v in share.items():
        p, vv = c.split("-")
        heat[int(vv), int(p)] = v

    fig = make_subplots(rows=2, cols=2, row_heights=[0.62, 0.38],
                        column_widths=[0.66, 0.34],
                        specs=[[{"colspan": 2}, None], [{}, {"type": "heatmap"}]],
                        subplot_titles=(f"{title}：{na}×{nb} 格着色",
                                        "各格价格路径", f"{na}×{nb} 时间占比（%）"))
    fig.add_trace(go.Scatter(x=close.index, y=close.values, mode="lines",
                             line=dict(color="#D5D8DC", width=1), name="价格",
                             showlegend=False), row=1, col=1)
    for i, c in enumerate(order):
        m = cell == c
        if not m.any():
            continue
        seg = close.where(m)
        fig.add_trace(go.Scatter(x=seg.index, y=seg.values, mode="lines+markers",
                                 line=dict(color=colors[i], width=2.5),
                                 marker=dict(size=4, color=colors[i]),
                                 name=f"{c}（{int(m.sum())} 天，{share[c]:.1f}%）"),
                      row=1, col=1)
    for i, c in enumerate(order):
        sub = close[cell == c]
        if len(sub) < 5:
            continue
        fig.add_trace(go.Scatter(x=sub.index, y=sub.values, mode="lines",
                                 line=dict(color=colors[i], width=1), name=c,
                                 showlegend=False), row=2, col=1)
    fig.add_trace(go.Heatmap(z=heat, x=[f"价格{i}" for i in range(na)],
                             y=[f"波动{j}" for j in range(nb)],
                             text=np.round(heat, 1), texttemplate="%{text}%",
                             colorscale="Blues", showscale=False), row=2, col=2)
    fig.update_layout(height=720, margin=dict(l=40, r=20, t=70, b=40),
                      hovermode="closest",
                      legend=dict(orientation="h", yanchor="bottom", y=1.06))
    return fig


def plot_animation(pk, vk, title):
    na, nb = sizes(pk, vk)
    cell = cells(pk, vk)
    d = pd.DataFrame({"格": cell})
    d["价格"] = d["格"].str.split("-").str[0].astype(int)
    d["波动"] = d["格"].str.split("-").str[1].astype(int)
    d["年份"] = d.index.year
    cnt = d.groupby(["年份", "价格", "波动"]).size().reset_index(name="天数")
    fig = px.scatter(cnt, x="价格", y="波动", size="天数", color="天数",
                     animation_frame="年份", size_max=45,
                     range_x=[-0.5, na - 0.5], range_y=[-0.5, nb - 0.5],
                     color_continuous_scale="Blues",
                     title=f"{title}：{na}×{nb} 占用随年份")
    fig.update_layout(height=460, margin=dict(l=40, r=20, t=60, b=40))
    fig.update_xaxes(tickmode="array", tickvals=list(range(na)),
                     ticktext=[f"价格{i}" for i in range(na)])
    fig.update_yaxes(tickmode="array", tickvals=list(range(nb)),
                     ticktext=[f"波动{j}" for j in range(nb)])
    return fig

## 一、短价格（Student-t, K=3）

In [4]:
plot_states('短价格').show()

## 二、短波动（Gaussian, K=3）

In [5]:
plot_states('短波动').show()

## 三、长价格（Student-t, K=2）

In [6]:
plot_states('长价格').show()

## 五、短/长维度的区分度

In [7]:
print(f"短价格 vs 长价格 一致率 = {agreement(fitted['短价格']['states'], fitted['长价格']['states']):.1%}")
print("（中长期不再建波动模型，所以没有波动维度的一致性）")

短价格 vs 长价格 一致率 = 48.5%
（中长期不再建波动模型，所以没有波动维度的一致性）


## 六、四个维度的画像与占比

In [8]:
for name, r in fitted.items():
    print(f"=== {name}（{r['model']}, K={K_OF[name]}）NLL={r['nll']:.4f} 熵={r['entropy']:.3f} ===")
    display(r["profile"])
    print("占比:", r["share"])
    print()

=== 短价格（Student-t(ν=5), K=3）NLL=1.2619 熵=0.079 ===


,ret_5,ret_10,er_10,snr_10
0,-1.40,-2.83,0.33,-0.82
1,0.86,1.21,0.15,0.38
2,2.24,5.02,0.54,1.50


占比: {0: 0.507, 1: 0.258, 2: 0.235}

=== 短波动（Gaussian, K=3）NLL=0.2903 熵=0.091 ===


,rv_5,atr_5,gk_10,abs_ret_5
0,12.91,1.16,10.63,0.67
1,20.48,1.84,17.03,1.05
2,36.28,3.20,31.36,2.03


占比: {0: 0.591, 1: 0.345, 2: 0.064}

=== 长价格（Student-t(ν=5), K=2）NLL=1.7567 熵=0.021 ===


,ret_60,ret_120,er_60,er_120,snr_60
0,-5.74,-4.45,0.12,0.07,-0.67
1,8.97,11.01,0.16,0.11,0.96


占比: {0: 0.553, 1: 0.447}



## 七、短周期九宫格（3×3）

In [9]:
plot_grid('短价格', '短波动', '短周期').show()

In [10]:
plot_animation('短价格', '短波动', '短周期').show()

## 八、中长期只有价格一维（K=2）

60–120 窗口下的"波动率状态"两个档位在图上几乎重合，没有可视化意义；
资金分析也只在短周期做。所以中长期不建波动模型，也就没有九宫格。

## 九、短周期九宫格的风险画像

In [11]:
from experiment.hmm_state.stage1 import risk_by, nonoverlap_check

cell = cells("短价格", "短波动")
display(risk_by(cell, px_panel))
print("非重叠复核:", nonoverlap_check(cell, px_panel))

,n,占比%,未来均值%,ES5,ES5_CI,MAE中位%,MDD中位%,MDD_P90%,可用
格,,,,,,,,,
0-0,302,29.3,-0.45,-9.95,"[-11.3, -7.6]",-4.24,-4.92,-10.97,True
0-1,182,17.7,0.45,-9.42,"[-11.6, -6.8]",-3.60,-4.34,-10.46,True
0-2,33,3.2,6.20,-1.67,样本不足,-2.27,-2.96,-6.11,True
1-0,155,15.1,-0.92,-11.83,"[-13.5, -8.4]",-3.91,-4.25,-8.42,True
1-1,105,10.2,-0.44,-12.72,"[-14.3, -6.0]",-3.31,-4.76,-9.37,True
1-2,6,0.6,4.60,1.79,样本不足,-0.35,-5.70,-6.60,False
2-0,164,15.9,0.50,-7.92,"[-8.4, -6.2]",-2.37,-4.22,-6.73,True
2-1,59,5.7,-0.27,-11.49,"[-11.6, -9.8]",-4.04,-4.72,-11.12,True
2-2,23,2.2,4.64,-1.69,样本不足,-1.62,-4.06,-10.39,False


非重叠复核: {'样本': 52, '格数': 2, 'ES5极差': 4.9}


## 十、资金变量

四类信号：期货基差（态度）、期货持仓（参与度/敞口，**不叫分歧度**）、ETF 份额、
两融（杠杆）、成交额（活跃度）。全部滚动 z 归一化，避免高波动期天然数值偏大。

In [12]:
z = load_signals(px_panel)
b = buckets(z)
display(coverage(z))


def plot_money(name):
    s = z[name].dropna()
    close = px_panel["close"].reindex(s.index)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.55, 0.45],
                        vertical_spacing=0.08,
                        subplot_titles=(f"{name}：{SIGNALS[name]}（滚动 z 分数）", "价格"))
    fig.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines",
                             line=dict(color="#8E44AD", width=1), name=name), row=1, col=1)
    fig.add_hline(y=0, line=dict(color="#BDC3C7", width=1, dash="dot"), row=1, col=1)
    fig.add_trace(go.Scatter(x=close.index, y=close.values, mode="lines",
                             line=dict(color="#34495E", width=1), name="价格"), row=2, col=1)
    fig.update_layout(height=480, margin=dict(l=40, r=20, t=60, b=40),
                      hovermode="x unified", showlegend=False)
    return fig

,信号,说明,口径,有效天,起始,截止
0,basis_z,基差（年化，次月均值）,态度,2758,2015-07-08,2026-11-13
1,oi_change_z,持仓变化（手）,绝对变化,2758,2015-07-08,2026-11-13
2,oi_change_pct_z,持仓变化率,相对变化,2758,2015-07-08,2026-11-13
3,margin_abs_z,两融余额变化（亿元）,绝对变化,3377,2012-12-14,2026-11-13
4,margin_pct_z,两融变化率,相对变化,3377,2012-12-14,2026-11-13
5,etf_abs_z,ETF 份额变化（亿份）,绝对变化,2638,2016-01-04,2026-11-13
6,etf_pct_z,ETF 份额变化率,相对变化,2514,2016-07-07,2026-11-13
7,turnover_z,成交额 / 20 日均额,相对变化,3378,2012-12-13,2026-11-13
8,oi_level_rel,持仓水平（相对 250 日均值）,水平,2639,2015-12-31,2026-11-13


### 各信号走势

In [13]:
for name in z.columns:
    plot_money(name).show()

### 各格子的资金 z 均值

先把 9 格 / 4 格的编号和各资金 z 并起来，看哪几个信号值得进条件转移分析。

In [14]:
d = pd.DataFrame({"格": cells("短价格", "短波动")}).join(z, how="inner")
display(d.groupby("格").mean().round(2))

,basis_z,oi_change_z,oi_change_pct_z,margin_abs_z,margin_pct_z,etf_abs_z,etf_pct_z,turnover_z,oi_level_rel
格,,,,,,,,,
0-0,-0.21,0.07,0.06,-0.24,-0.24,0.11,-0.03,-0.47,-0.36
0-1,-0.19,-0.01,-0.02,-0.68,-0.67,0.36,0.17,-0.45,0.47
0-2,-1.23,-0.17,-0.18,-2.58,-2.59,1.98,1.78,-0.12,1.27
1-0,-0.05,0.00,0.01,0.23,0.21,0.07,-0.03,0.12,-0.40
1-1,0.06,-0.10,-0.09,0.28,0.21,-0.19,-0.36,0.00,0.47
1-2,0.47,-0.62,-0.54,-0.06,-0.12,0.92,0.80,0.38,0.80
2-0,0.12,0.01,0.01,0.79,0.68,-0.20,-0.25,0.60,0.03
2-1,-0.07,0.06,0.05,0.72,0.61,-0.45,-0.57,0.67,0.26
2-2,1.07,0.13,0.11,2.29,2.23,0.42,0.17,1.93,0.16
